In [2]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from utils import DEVICE, SUPERCLASSES, preprocess_signals, extract_features, load_preprocessed, \
    PTBXLDataset, train_one_epoch, evaluate, tune_thresholds, make_warmup_cosine
from baseline import NUM_CLASSES, make_loaders, run_xgboost, train_torch_baseline,CNN, ResNet1D, Transformer
from proposed_model import CnnTransformer


ModuleNotFoundError: No module named 'wfdb'

# Data Preprocessing

In [ ]:
# preprocess signal and hand-crafted features
RUN_FEATURE_EXTRACTION = True  
preprocess_signals()
if RUN_FEATURE_EXTRACTION:
    extract_features()

# Baselines

In [ ]:
def run_baseline(model, title, checkpoint_dir, *, augment_train, epochs, lr,
                 patience, warmup_epochs=0, weight_decay=1e-4, grad_clip=1.0,
                 banner=None):
    print("\n" + "=" * 60)
    print(banner or f"BASELINE — {title}")
    print("=" * 60)
    cfg = {"checkpoint_dir": checkpoint_dir, "epochs": epochs, "lr": lr,
           "weight_decay": weight_decay, "patience": patience,
           "grad_clip": grad_clip, "warmup_epochs": warmup_epochs}
    loaders = make_loaders(augment_train=augment_train)
    return train_torch_baseline(model, cfg, loaders, title)

BASELINES = {
    "cnn": dict(
        model=lambda: CNN(num_classes=NUM_CLASSES),
        title="1D CNN", banner="Baseline 2 - 1D CNN",
        checkpoint_dir="./checkpoint_cnn", augment_train=False,
        epochs=30, lr=1e-3, weight_decay=1e-4, patience=7, warmup_epochs=0),
    "resnet1d": dict(
        model=lambda: ResNet1D(num_classes=NUM_CLASSES),
        title="ResNet1D", banner="Baseline 3 — ResNet1D",
        checkpoint_dir="./checkpoint_resnet", augment_train=True,
        epochs=40, lr=1e-3, weight_decay=1e-4, patience=8, warmup_epochs=0),
    "transformer": dict(
        model=lambda: Transformer(num_classes=NUM_CLASSES),
        title="Transformer", banner="Baseline 4 — Transformer",
        checkpoint_dir="./checkpoint_transformer", augment_train=True,
        epochs=40, lr=3e-4, weight_decay=1e-4, patience=10, warmup_epochs=4),
}


In [ ]:
# === Run all baselines and summarize ===
print(f"Device: {DEVICE}\n")

results = {"XGBoost": run_xgboost()}        
for name, spec in BASELINES.items():
    spec = dict(spec)
    spec["model"] = spec["model"]()           # instantiate a fresh model
    results[name] = run_baseline(**spec)

print("\n" + "=" * 60)
print("Baseline Summary (test set)")
print("=" * 60)
print(f"  {'Model':20s}  {'Macro F1':>9s}  {'Macro AUROC':>12s}")
for name, m in results.items():
    print(f"  {name:20s}  {m['f1']:>9.4f}  {m['auroc']:>12.4f}")


# Proposed Model: CNN + Transformer

In [ ]:
# Proposed model: CNN + Transformer
CHECKPOINT_DIR = "./checkpoint_proposed"
BEST_MDL       = os.path.join(CHECKPOINT_DIR, "best.pt")
LAST_MDL       = os.path.join(CHECKPOINT_DIR, "last.pt")
HISTORY_PATH   = os.path.join(CHECKPOINT_DIR, "history.json")

# Architecture
LEAD_DIM = 32    # per-lead feature dim after grouped CNN
D_MODEL = 192   # temporal transformer dim
NHEAD = 4
NUM_LAYERS = 3
DIM_FEEDFORWARD = 384
LEAD_NHEAD = 4
LEAD_LAYERS = 1
DROPOUT = 0.15

# Training
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 4
EARLY_STOP_PATIENCE = 12
GRAD_CLIP = 1.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print("Proposed model: CNN + Transformer")
print("Loading preprocessed data...")
X_train, y_train, X_val, y_val, X_test, y_test = load_preprocessed()
print(f"  Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

train_loader = DataLoader(PTBXLDataset(X_train, y_train, augment=True),
                            batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=(DEVICE == "cuda"))
val_loader   = DataLoader(PTBXLDataset(X_val, y_val, augment=False),
                            batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=(DEVICE == "cuda"))
test_loader  = DataLoader(PTBXLDataset(X_test, y_test, augment=False),
                            batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=(DEVICE == "cuda"))

print("\nInitializing model...")
model = CnnTransformer(num_classes=len(SUPERCLASSES)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Parameters: {n_params:,}")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(),
                                lr=LEARNING_RATE,
                                weight_decay=WEIGHT_DECAY)
scheduler = make_warmup_cosine(optimizer, WARMUP_EPOCHS, EPOCHS)

history = {"train_loss": [], "train_f1": [],
            "val_loss":   [], "val_f1":   [],
            "val_precision": [], "val_recall": [], "val_auroc": [],
            "lr": []}

best_auroc = 0.0
patience = 0

print("\nStarting training...")
for epoch in range(EPOCHS):
    train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
    val_metrics, _, _    = evaluate(model, val_loader, criterion)
    lr_now = scheduler.get_last_lr()[0]
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_metrics["loss"])
    history["val_f1"].append(val_metrics["f1"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])
    history["val_auroc"].append(val_metrics["auroc"])
    history["lr"].append(lr_now)

    print(f"\nEpoch {epoch + 1:02d}/{EPOCHS}  (lr={lr_now:.2e})")
    print(f"  Train  Loss={train_loss:.4f}  F1={train_f1:.4f}")
    print(f"  Val    Loss={val_metrics['loss']:.4f}  "
            f"F1={val_metrics['f1']:.4f}  "
            f"Prec={val_metrics['precision']:.4f}  "
            f"Recall={val_metrics['recall']:.4f}  "
            f"AUROC={val_metrics['auroc']:.4f}")
    print(f"  Val per-class AUROC: " + "  ".join(
        f"{cls}={v:.3f}" for cls, v in val_metrics["per_class_auroc"].items()
    ))

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_f1": val_metrics["f1"],
        "val_auroc": val_metrics["auroc"],
    }, LAST_MDL)

    if val_metrics["auroc"] > best_auroc:
        best_auroc = val_metrics["auroc"]
        patience = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_auroc": val_metrics["auroc"],
        }, BEST_MDL)
        print(f"  ✓ Best model saved (val AUROC: {best_auroc:.4f})")
    else:
        patience += 1
        print(f"  Patience {patience}/{EARLY_STOP_PATIENCE}")
        if patience >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch + 1}")
            break

    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

# Load best
print("\nLoading best model for evaluation...")
ckpt = torch.load(BEST_MDL, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])

# Threshold tuning
print("\nTuning per-class thresholds on validation set...")
thresholds = tune_thresholds(model, val_loader, len(SUPERCLASSES))
print("  Tuned thresholds:")
for cls, t in zip(SUPERCLASSES, thresholds):
    print(f"    {cls:5s}: {t:.3f}")
np.save(os.path.join(CHECKPOINT_DIR, "thresholds.npy"), thresholds)

# Test
print("\nTest with default threshold (0.5):")
test_default, _, _ = evaluate(model, test_loader, criterion, thresholds=None)
print(f"  Macro F1={test_default['f1']:.4f}  "
        f"Prec={test_default['precision']:.4f}  "
        f"Recall={test_default['recall']:.4f}")

print("\nTest with tuned per-class thresholds:")
test_metrics, test_preds, test_targs = evaluate(model, test_loader, criterion,
                                                    thresholds=thresholds)

print("\n" + "=" * 60)
print("Test Results")
print("=" * 60)
print(f"  Loss      : {test_metrics['loss']:.4f}")
print(f"  Macro F1  : {test_metrics['f1']:.4f}")
print(f"  Macro Prec: {test_metrics['precision']:.4f}")
print(f"  Macro Rec : {test_metrics['recall']:.4f}")
print(f"  Macro AUC : {test_metrics['auroc']:.4f}")
print("  Per-class AUROC:")
for cls, auc in test_metrics["per_class_auroc"].items():
    print(f"    {cls:5s}: {auc:.4f}")

np.save(os.path.join(CHECKPOINT_DIR, "test_preds.npy"), test_preds)
np.save(os.path.join(CHECKPOINT_DIR, "test_targs.npy"), test_targs)
with open(os.path.join(CHECKPOINT_DIR, "test_metrics.json"), "w") as f:
    json.dump({k: v for k, v in test_metrics.items()
                if k != "per_class_auroc"} | {"per_class_auroc": test_metrics["per_class_auroc"]},
                f, indent=2)
print(f"\nArtifacts saved to {CHECKPOINT_DIR}/")